# Full directed weighted network analysis

This read-only Stage 3 audit analyses the complete available interaction network. It retains the graph as directed, uses edge weights, preserves self-loops for analysis, and does not select a 10K subset or construct a diffusion model.

## 1. Load network data

In [ ]:
from pathlib import Path
import json
import math

import networkx as nx
from networkx.algorithms.link_analysis.pagerank_alg import _pagerank_python
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data' / 'raw' / 'network_edges.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the project root or its notebooks directory.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
for directory in [FIGURES_DIR, TABLES_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Raw inputs are only read; all calculations below use in-memory dataframes.
edges = pd.read_csv(RAW_DIR / 'network_edges.csv')
print(f'Loaded {len(edges):,} directed edge rows')
print(edges[['Source', 'Target', 'Weight']].head().to_string(index=False))

def save_bar_chart(labels, values, title, filename, color=(55,126,184)):
    width, height, margin = 1200, 680, 85
    image = Image.new('RGB', (width, height), 'white')
    draw, font = ImageDraw.Draw(image), ImageFont.load_default()
    minimum, maximum = min(0, min(values)) if values else 0, max(0, max(values)) if values else 1
    value_range = maximum - minimum or 1
    plot_width, plot_height = width - 2 * margin, height - 2 * margin
    draw.text((margin, 25), title, fill='black', font=font)
    zero_y = int(height-margin-((0-minimum)/value_range)*(plot_height-45))
    draw.line((margin, zero_y, width-margin, zero_y), fill='black', width=2)
    draw.line((margin, margin, margin, height-margin), fill='black', width=2)
    slot, bar_width = plot_width / max(len(values), 1), max(2, int(plot_width / max(len(values), 1) * 0.70))
    for i, (label, value) in enumerate(zip(labels, values)):
        x0 = int(margin + i * slot + (slot-bar_width)/2)
        y = int(height-margin-((value-minimum)/value_range)*(plot_height-45))
        draw.rectangle((x0, min(y, zero_y), x0+bar_width, max(y, zero_y)), fill=color)
        if len(values) <= 25:
            draw.text((x0, max(margin+22, y-15)), f'{value:.3g}', fill='black', font=font)
            draw.text((x0, height-margin+8), str(label)[:18], fill='black', font=font)
    output = FIGURES_DIR / filename
    image.save(output)
    print(f'Saved figure: {output.relative_to(PROJECT_ROOT)}')

def save_histogram(values, title, filename, bins=40, log1p=False):
    series = pd.Series(values).dropna().astype(float)
    if log1p:
        series = series.map(math.log1p)
    lower, upper = series.min(), series.max()
    if lower == upper:
        counts, labels = [len(series)], [f'{lower:.2f}']
    else:
        step = (upper-lower)/bins
        counts = [0]*bins
        for value in series:
            counts[min(int((value-lower)/step), bins-1)] += 1
        labels = [f'{lower+i*step:.2f}' for i in range(bins)]
    save_bar_chart(labels, counts, title, filename, color=(77,175,74))


In [ ]:
edge_summary = {
    'edges': int(len(edges)),
    'unique_source_nodes': int(edges['Source'].nunique()),
    'unique_target_nodes': int(edges['Target'].nunique()),
    'total_unique_nodes': int(len(set(edges['Source']) | set(edges['Target']))),
    'self_loops': int((edges['Source'] == edges['Target']).sum()),
    'minimum_edge_weight': float(edges['Weight'].min()),
    'maximum_edge_weight': float(edges['Weight'].max()),
    'mean_edge_weight': float(edges['Weight'].mean()),
    'median_edge_weight': float(edges['Weight'].median())
}
print(json.dumps(edge_summary, indent=2))

## 2. Build directed graph

In [ ]:
# Each row is Source -> Target. DiGraph preserves direction; Weight is retained for weighted measures.
# Self-loops remain in G so their impact can be measured rather than silently discarded.
G = nx.from_pandas_edgelist(edges, source='Source', target='Target', edge_attr='Weight', create_using=nx.DiGraph())
weak_components = list(nx.weakly_connected_components(G))
strong_components = list(nx.strongly_connected_components(G))
largest_wcc_size = max(map(len, weak_components))
largest_scc_size = max(map(len, strong_components))
graph_summary = {
    'nodes': G.number_of_nodes(), 'edges': G.number_of_edges(), 'density': nx.density(G),
    'weakly_connected_components': len(weak_components), 'largest_weak_component_size': largest_wcc_size,
    'strongly_connected_components': len(strong_components), 'largest_strong_component_size': largest_scc_size
}
print(json.dumps(graph_summary, indent=2))
print('Self-loops are retained in this audit. For later diffusion modelling, compare sensitivity with and without them; they may represent self-interaction but can also inflate self-reinforcement.')

## 3. Degree analysis

In [ ]:
nodes = list(G.nodes())
node_features = pd.DataFrame({'node_id': nodes})
node_features['in_degree'] = node_features['node_id'].map(dict(G.in_degree())).astype(int)
node_features['out_degree'] = node_features['node_id'].map(dict(G.out_degree())).astype(int)
node_features['total_degree'] = node_features['in_degree'] + node_features['out_degree']
node_features['weighted_in_degree'] = node_features['node_id'].map(dict(G.in_degree(weight='Weight'))).astype(float)
node_features['weighted_out_degree'] = node_features['node_id'].map(dict(G.out_degree(weight='Weight'))).astype(float)
node_features['weighted_degree'] = node_features['weighted_in_degree'] + node_features['weighted_out_degree']
degree_summary = node_features[['in_degree','out_degree','total_degree','weighted_in_degree','weighted_out_degree','weighted_degree']].describe().T
print(degree_summary.to_string())
save_histogram(node_features['in_degree'], 'In-degree distribution (log1p degree)', 'in_degree_distribution.png', log1p=True)
save_histogram(node_features['out_degree'], 'Out-degree distribution (log1p degree)', 'out_degree_distribution.png', log1p=True)
save_histogram(node_features['weighted_degree'], 'Weighted-degree distribution (log1p degree)', 'weighted_degree_distribution.png', log1p=True)

## 4. Weighted PageRank

In [ ]:
# Use NetworkX's pure-Python PageRank implementation because this runtime has no SciPy backend.
pagerank = _pagerank_python(G, alpha=0.85, weight='Weight', max_iter=200, tol=1e-10)
node_features['pagerank'] = node_features['node_id'].map(pagerank).astype(float)
top_pagerank = node_features.sort_values('pagerank', ascending=False).head(20).reset_index(drop=True)
print(top_pagerank.to_string(index=False))
top_pagerank_path = TABLES_DIR / 'top_pagerank_users.csv'
top_pagerank.to_csv(top_pagerank_path, index=False)
print(f'Saved {top_pagerank_path.relative_to(PROJECT_ROOT)}')

## 5. Normalization

In [ ]:
def min_max(series):
    minimum, maximum = series.min(), series.max()
    return pd.Series(0.0, index=series.index) if minimum == maximum else (series-minimum)/(maximum-minimum)

node_features['degree_norm'] = min_max(node_features['total_degree'])
node_features['weighted_degree_norm'] = min_max(node_features['weighted_degree'])
node_features['pagerank_norm'] = min_max(node_features['pagerank'])
normalization_ranges = node_features[['degree_norm','weighted_degree_norm','pagerank_norm']].agg(['min','max'])
print(normalization_ranges.to_string())
assert ((node_features[['degree_norm','weighted_degree_norm','pagerank_norm']] >= 0) & (node_features[['degree_norm','weighted_degree_norm','pagerank_norm']] <= 1)).all().all()

## 6. Trust score

In [ ]:
# Project modelling feature following the requested reference-paper formulation; it is not real-world trust.
node_features['trust_score'] = 0.6 * node_features['degree_norm'] + 0.4 * node_features['pagerank_norm']
assert ((node_features['trust_score'] >= 0) & (node_features['trust_score'] <= 1)).all()
top_trust = node_features.sort_values('trust_score', ascending=False).head(20).reset_index(drop=True)
print(top_trust.to_string(index=False))
top_trust_path = TABLES_DIR / 'top_trust_users.csv'
top_trust.to_csv(top_trust_path, index=False)
save_histogram(node_features['trust_score'], 'Trust-score distribution', 'trust_score_distribution.png')
print(f'Saved {top_trust_path.relative_to(PROJECT_ROOT)}')

## 7. Connect network with tweet data

In [ ]:
tweets = pd.read_csv(RAW_DIR / 'master_tweets.csv')
tweet_users = set(tweets['User'].astype(str))
network_nodes = set(node_features['node_id'].astype(str))
overlapping_users = tweet_users & network_nodes
overlap_summary = {
    'tweet_users': len(tweet_users), 'network_nodes': len(network_nodes), 'overlapping_users': len(overlapping_users),
    'tweet_users_represented_in_network_pct': 100*len(overlapping_users)/len(tweet_users),
    'network_nodes_represented_in_tweets_pct': 100*len(overlapping_users)/len(network_nodes)
}
print(json.dumps(overlap_summary, indent=2))

# Target values are kept as labels; this aggregation intentionally makes no semantic inference.
tweet_aggregates = tweets.groupby('User').agg(tweet_count=('Tweet','size'), total_likes=('Likes','sum'), total_retweets=('Retweets','sum'), avg_likes=('Likes','mean'), avg_retweets=('Retweets','mean'))
target_counts = pd.crosstab(tweets['User'], tweets['Target']).reindex(columns=[0.0, 1.0], fill_value=0).rename(columns={0.0:'target_0_count',1.0:'target_1_count'})
tweet_metrics = tweet_aggregates.join(target_counts, how='left')
tweet_metrics['target_0_ratio'] = tweet_metrics['target_0_count'] / tweet_metrics['tweet_count']
tweet_metrics['target_1_ratio'] = tweet_metrics['target_1_count'] / tweet_metrics['tweet_count']
print(tweet_metrics.head().to_string())

## 8. Full node feature table

In [ ]:
network_node_features_full = node_features.set_index('node_id').join(tweet_metrics, how='inner').reset_index()
ordered_columns = ['node_id','in_degree','out_degree','total_degree','weighted_in_degree','weighted_out_degree','weighted_degree','pagerank','degree_norm','weighted_degree_norm','pagerank_norm','trust_score','tweet_count','total_likes','total_retweets','avg_likes','avg_retweets','target_0_count','target_1_count','target_0_ratio','target_1_ratio']
network_node_features_full = network_node_features_full[ordered_columns].sort_values('node_id').reset_index(drop=True)
feature_path = PROCESSED_DIR / 'network_node_features_full.csv'
network_node_features_full.to_csv(feature_path, index=False)
print(f'Feature rows (network and tweet overlap): {len(network_node_features_full):,}')
print(f'Saved {feature_path.relative_to(PROJECT_ROOT)}')
network_node_features_full.head()

## 9. Network visualizations

In [ ]:
save_histogram(node_features['pagerank'], 'PageRank distribution (log1p value)', 'pagerank_distribution.png', log1p=True)
top_pr_plot = top_pagerank.sort_values('pagerank')
save_bar_chart(top_pr_plot['node_id'].tolist(), top_pr_plot['pagerank'].tolist(), 'Top 20 PageRank nodes', 'top_pagerank_users.png', color=(228,26,28))
top_trust_plot = top_trust.sort_values('trust_score')
save_bar_chart(top_trust_plot['node_id'].tolist(), top_trust_plot['trust_score'].tolist(), 'Top 20 trust-score nodes', 'top_trust_users.png', color=(152,78,163))

def save_scatter(x_values, y_values, title, filename):
    width, height, margin = 1200, 680, 85
    image = Image.new('RGB', (width,height), 'white')
    draw, font = ImageDraw.Draw(image), ImageFont.load_default()
    x_min, x_max, y_min, y_max = min(x_values), max(x_values), min(y_values), max(y_values)
    x_range, y_range = x_max-x_min or 1, y_max-y_min or 1
    draw.text((margin,25), title, fill='black', font=font)
    draw.line((margin,height-margin,width-margin,height-margin),fill='black',width=2)
    draw.line((margin,margin,margin,height-margin),fill='black',width=2)
    for x, y in zip(x_values, y_values):
        px = int(margin + (x-x_min)/x_range*(width-2*margin))
        py = int(height-margin - (y-y_min)/y_range*(height-2*margin))
        draw.ellipse((px-1,py-1,px+1,py+1), fill=(55,126,184))
    output = FIGURES_DIR / filename
    image.save(output)
    print(f'Saved figure: {output.relative_to(PROJECT_ROOT)}')

save_scatter(node_features['total_degree'].map(math.log1p), node_features['pagerank'].map(math.log1p), 'Degree vs PageRank (both log1p)', 'degree_pagerank_relationship.png')
pearson_correlation = node_features['total_degree'].corr(node_features['pagerank'], method='pearson')
# Spearman correlation equals Pearson correlation after average-rank transformation.
spearman_correlation = node_features['total_degree'].rank(method='average').corr(node_features['pagerank'].rank(method='average'), method='pearson')
print(f'Pearson correlation (degree, PageRank): {pearson_correlation:.4f}')
print(f'Spearman correlation (degree, PageRank): {spearman_correlation:.4f}')
print('These correlations describe association only and do not establish causation.')

## 10. Top users table

In [ ]:
tweet_count_by_node = node_features.set_index('node_id').join(tweet_metrics[['tweet_count']], how='left')['tweet_count'].fillna(0)
ranking_specs = {'pagerank':'pagerank', 'trust_score':'trust_score', 'weighted_degree':'weighted_degree'}
ranked_frames = []
for label, column in ranking_specs.items():
    ranked = node_features.nlargest(20, column).copy()
    ranked.insert(0, 'ranking_metric', label)
    ranked.insert(1, 'rank', range(1, len(ranked)+1))
    ranked_frames.append(ranked[['ranking_metric','rank','node_id',column]].rename(columns={column:'metric_value'}))
tweet_ranked = tweet_metrics.nlargest(20, 'tweet_count').reset_index().rename(columns={'User':'node_id'})
tweet_ranked.insert(0, 'ranking_metric', 'tweet_count')
tweet_ranked.insert(1, 'rank', range(1, len(tweet_ranked)+1))
ranked_frames.append(tweet_ranked[['ranking_metric','rank','node_id','tweet_count']].rename(columns={'tweet_count':'metric_value'}))
network_top_users = pd.concat(ranked_frames, ignore_index=True)
network_top_users_path = TABLES_DIR / 'network_top_users.csv'
network_top_users.to_csv(network_top_users_path, index=False)
print(network_top_users.to_string(index=False))
print(f'Saved {network_top_users_path.relative_to(PROJECT_ROOT)}')

## 11. Data-quality checks

In [ ]:
source_nodes = set(edges['Source'].astype(str))
target_nodes = set(edges['Target'].astype(str))
quality_checks = {
    'source_nodes_missing_from_graph': len(source_nodes - network_nodes),
    'target_nodes_missing_from_graph': len(target_nodes - network_nodes),
    'tweet_users_absent_from_network': len(tweet_users - network_nodes),
    'network_nodes_absent_from_tweets': len(network_nodes - tweet_users),
    'self_loops': int((edges['Source'] == edges['Target']).sum()),
    'zero_weight_edges': int((edges['Weight'] == 0).sum()),
    'negative_weight_edges': int((edges['Weight'] < 0).sum()),
    'duplicate_source_target_pairs': int(edges.duplicated(subset=['Source','Target']).sum())
}
print(json.dumps(quality_checks, indent=2))
print('No rows are removed by these checks. Self-loops require a documented modelling decision in a later stage.')

## 12. Research summary

In [ ]:
research_summary = {
    'total_network_nodes': int(G.number_of_nodes()), 'total_edges': int(G.number_of_edges()),
    'overlap_users': int(len(overlapping_users)), 'graph_density': float(nx.density(G)),
    'largest_weak_component_size': int(largest_wcc_size), 'top_pagerank_value': float(node_features['pagerank'].max()),
    'top_trust_score': float(node_features['trust_score'].max()),
    'degree_statistics': degree_summary[['mean','50%','max']].to_dict(),
    'self_loop_count': int(quality_checks['self_loops']),
    'important_limitations': [
        'The graph is directed and static; weights do not include interaction timestamps.',
        'Self-loops are retained for audit and need sensitivity analysis before diffusion modelling.',
        'Tweet coverage is incomplete for network nodes, and network coverage is incomplete for tweet users.',
        'The project trust_score is a modelling feature, not evidence of real-world trust.'
    ]
}
print(json.dumps(research_summary, indent=2))
print('\nThe interaction network is static because network_edges.csv does not contain interaction timestamps. Temporal variation will therefore be represented through evolving tweet-derived node features rather than timestamped edge snapshots.')